# Lab 06 — Orchestration: ReAct, Plan-and-Execute, DAGs

Kiel University · Agentic AI (infAgAI-01a) · Winter 2026

**Learning objectives** — after this lab you can:

- explain **orchestration** as the control layer of an agentic system and answer the session's
  one question — *who decides the next step, your code or the model?* — for any design,
- implement a **ReAct agent from scratch** (Thought / Action / Observation, parsed from plain
  text, exactly like Yao et al. before function-calling APIs existed),
- **instrument** the loop with a trace and watch it loop, derail, and recover — and build a
  **loop detector** against near-duplicate actions,
- implement a **Plan-and-Execute** variant with an explicit plan artifact, a narrow-context
  executor, and **replanning as an observable event**,
- run both regimes on the same tasks and compare **step counts, token cost, latency shape and
  failure modes**,
- wire a fixed subtask as a **static DAG** and read off parallelism and termination from the
  graph structure.

> ⏱️ Estimated time: 90–120 minutes. All tools in this lab are **offline** — a tiny synthetic
> "web" in `data/corpus.json` stands in for live search, so only local Ollama is required.

## Theory recap — who decides the next step?

### Orchestration, defined

**Orchestration** is the **control layer** of an agentic system: the logic that determines
which step executes next, and with what input. Distinguish it from the **computation** inside
the steps — LLM calls and tools do the work; orchestration sequences it (sequencing, branching,
looping, termination). The session's one question, worth carrying through your career: **who
decides the next step — your code or the model?** Code-owned control maximises
*predictability*; model-owned control maximises *flexibility* — and the two cannot be maximal
at once, as a matter of logic, not technology. Between the poles lies a spectrum: fixed chains,
branching workflows, planned runs, fully dynamic loops. This is Anthropic's workflow/agent
distinction from S01/S02 with higher resolution: *workflows follow code-defined paths; agents
direct their own process*.

### ReAct — interleaved reasoning and acting

**ReAct** (Yao et al., ICLR 2023) is the canonical dynamic pattern: the model alternates a
**Thought** (free-text reflection — what do I know, what is missing, what next? — never
executed), an **Action** (a tool call — `search`, `fetch`, `finish` — emitted as text), and an
**Observation** (the tool result, appended to context before the next thought). Historically it
was a *prompt trick*: few-shot example traces taught the format, and the runtime parsed action
lines by string matching — no function-calling APIs existed in 2022. Your S02 agent loop is
ReAct, productised. The ablations explain why interleaving wins: **reason-only** (CoT) runs on
parametric memory alone and hallucinates confidently; **act-only** cannot decompose the goal or
change course. The synergy, as a chiasmus: **thoughts ground actions in a plan; observations
ground thoughts in the world.** The failure modes are structural: **myopia** (every decision one
step deep, no global plan), **looping** (re-issuing a failed search with minor rewording),
**derailment** (one misleading observation bends the trajectory), **context growth** (the trace
is the agent's working memory — and its growing bill). Mitigations: step caps, loop detection,
compaction (S08) — or: plan first.

### Plan-and-Execute — plan first, act cheaply, replan on surprise

**Plan-and-Execute** front-loads the decisions: a **planner** (your strongest model, S05) emits
a complete step list upfront — the **plan as artifact**, which you can log, inspect, approve and
test. An **executor** runs one step at a time with **narrow context** (its step and its inputs,
not the history) — often a cheaper model, sometimes plain code. **Replanning is an explicit,
observable event**, triggered by step failure, a surprising observation, or a budget threshold —
not silent drift. Cost shape: few heavy planner calls bracket many light executor calls (the S05
barbell). Weakness: **plans go stale** in volatile environments, and constant replanning pays
planner prices for ReAct behaviour. Evidence for the idea: Plan-and-Solve prompting (Wang et
al., 2023) — planning before solving beats plain step-by-step.

### Static workflows and DAGs

At the static end, code owns the control flow entirely: a **DAG** (directed acyclic graph) whose
nodes are steps and whose edges are data dependencies. Execution follows **topological order**;
nodes not ordered by dependencies are provably independent and run **in parallel for free**.
*Static* means the graph is fixed at design time — the LLM fills node content but has no vote on
which node runs next. **Acyclicity guarantees termination by construction**: a ReAct loop is a
cyclic graph, and that cycle is exactly what makes it powerful and risky. Anthropic's five
workflow patterns are the vocabulary of this range: **prompt chaining** (fixed sequence with
programmatic gates), **routing** (a classifier dispatches to specialised branches),
**parallelization** (*sectioning*: independent subtasks concurrently; *voting*: same task N
times, aggregated), **orchestrator-workers** (an LLM decomposes at run time, workers execute,
code merges — the bridge toward agents), **evaluator-optimizer** (generate–critique loop, works
when critique is easier than creation).

### Choosing a regime

No regime dominates — **the task's predictability decides, not fashion**. Static DAG: highest
predictability, fixed cost, unit-testable nodes, only coded branches. Plan-and-Execute: plan
visible upfront, adapts at replan points, cost estimable once the plan exists. ReAct: adapts to
every observation, path and cost emerge at run time — *unbounded without caps* — and bugs become
statistical, so tracing is mandatory (S11). Anthropic's rule: **use the simplest pattern that
works; add autonomy only when measured.**

### This lab

Exactly what the lecture's lab link announced: you **instrument a ReAct loop on the lab task and
watch it loop, derail, and recover** — then build the Plan-and-Execute variant on the same task,
compare traces, step counts and token cost, and wire a fixed subtask as a small static DAG.

## Part A — Setup & Ollama connectivity

One tool-capable local model is enough for this lab (default `qwen2.5:7b`, configurable via
`OLLAMA_MODEL`). We deliberately do **not** use the function-calling API today: the point of
Part C is to rebuild ReAct the way the paper did — plain text, few-shot format, string
parsing — so you see what the platforms later productised (S03).

In [ ]:
import os
import re
import json
import time
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MODEL = os.environ.get("OLLAMA_MODEL", "qwen2.5:7b")

OLLAMA_OK = False
try:
    import ollama
    ollama.list()
    OLLAMA_OK = True
    print(f"Ollama is running — using model {MODEL!r}.")
except Exception as exc:
    print("Could not reach Ollama:", exc)
    print("→ Start Ollama with `ollama serve` and pull the model with `ollama pull qwen2.5:7b`.")


def llm(messages, temperature=0.2, num_predict=512, stop=None):
    """One chat call, returning (text, stats). All LLM cost accounting runs through here."""
    options = {"temperature": temperature, "num_predict": num_predict}
    if stop:
        options["stop"] = stop
    t0 = time.time()
    resp = ollama.chat(model=MODEL, messages=messages, options=options)
    msg = resp["message"] if isinstance(resp, dict) else resp.message
    content = (msg["content"] if isinstance(msg, dict) else msg.content) or ""
    get = resp.get if isinstance(resp, dict) else lambda k, d=0: getattr(resp, k, d)
    stats = {"prompt_tokens": get("prompt_eval_count", 0) or 0,
             "output_tokens": get("eval_count", 0) or 0,
             "latency_s": round(time.time() - t0, 2)}
    return content.strip(), stats


if OLLAMA_OK:
    text, stats = llm([{"role": "user", "content": "Reply with exactly: READY"}], num_predict=16)
    print(text, "|", stats)

> **Q:** Define *orchestration* in an agentic system and distinguish it from the computation layer.
<details><summary>Click for answer</summary>

Orchestration is the <b>control layer</b>: the logic that decides which step executes next, with
what input — sequencing, branching, looping, termination. The computation layer is the work
inside the steps: LLM calls, tool invocations, plain functions. The distinction matters because
two systems with identical components but different control regimes have radically different
predictability, cost, and debuggability profiles.
</details>

## Part B — The offline research "web" and its tools

The build provides `data/corpus.json`: **8 synthetic pages** about the EU AI Act — the lecture's
claim-verification example. The corpus is deliberately booby-trapped:

- a **primary source** (`europa.example/ai-act/timeline`) with the official dates,
- a **news summary** that points to the primary source,
- a **misleading blog** claiming the GPAI obligations were "delayed until 2027" (derailment bait),
- pages that talk *around* one of our tasks without answering it (loop bait).

On top of it we build the research agent's tool set for today: `search(query)` (keyword scoring,
top-k results), `fetch(url)` (full page text), and — for ReAct — a `finish(answer)` action. All
offline and deterministic, so differences between runs come from **orchestration**, not from a
changing web.

In [ ]:
DATA_PATH = "data/corpus.json"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    corpus = json.load(___)                 # parse the opened JSON file object

print(f"{len(corpus)} pages loaded:")
for page in corpus:
    print(f"  {page['url']:<50} {page['title'][:60]}")

<details>
<summary><b>Click here for the solution</b></summary>

```python
with open(DATA_PATH, "r", encoding="utf-8") as f:
    corpus = json.load(f)
```

</details>

In [ ]:
STOPWORDS = set(("the a an of to in on for and or is are was were be been when does do did "
                 "what which who with by from under it its this that").split())


def tokenize(text):
    return [w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOPWORDS]


def search(query, k=3):
    """Keyword search over the corpus: score = number of shared query terms."""
    q_tokens = set(tokenize(query))
    hits = []
    for page in corpus:
        page_tokens = set(tokenize(page["title"] + " " + page["text"]))
        score = len(q_tokens & ___)          # overlap between query terms and page terms
        if score > 0:
            hits.append((score, {"url": page["url"], "title": page["title"],
                                 "snippet": page["text"][:110] + "…"}))
    hits.sort(key=lambda h: -h[0])
    hits = [h[1] for h in hits]
    return hits[:___]                        # only the top-k results reach the agent


def fetch(url):
    """Return the full text of a page, or a 404 error string."""
    for page in corpus:
        if page["url"] == ___:               # exact URL match
            return page["text"]
    return f"ERROR 404: no page at {url!r}"


# self-test (no LLM needed)
top = search("EU AI Act GPAI obligations application date")
assert top and top[0]["url"] == "europa.example/ai-act/timeline", top
assert "2 August 2025" in fetch("europa.example/ai-act/timeline")
assert fetch("nonsense.example/nope").startswith("ERROR 404")
print("Tool self-test passed. Top hit:", top[0]["url"])

<details>
<summary><b>Click here for the solution</b></summary>

```python
        score = len(q_tokens & page_tokens)
    ...
    return hits[:k]
    ...
        if page["url"] == url:
```

</details>

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `search` is a deliberately crude bag-of-words scorer: shared-term count between query and
  page, stopwords removed. Crude is a feature — imperfect retrieval is what makes the agent's
  *orchestration* decisions (reformulate? fetch? give up?) meaningful.
- `fetch` is exact-match on URL and returns an **error string** rather than raising: agents live
  on observations, so tool failures must arrive *as text in context*, where the next thought (or
  the replanner) can react to them.
- Everything is deterministic. When runs differ later, the difference is the control regime.

</details>

In [ ]:
# The lab task suite: two verifiable claims + one deliberate trap.
TASKS = [
    {"id": "gpai_date",
     "question": ("When do the obligations for providers of general-purpose AI (GPAI) models "
                  "under the EU AI Act start to apply? Give the exact date."),
     "expected": "2 August 2025"},
    {"id": "max_fine",
     "question": ("What is the maximum administrative fine under the EU AI Act for violating "
                  "the prohibited-practices rules of Article 5? Give the amount."),
     "expected": "35"},
    {"id": "loop_trap",
     "question": ("Which EU member state was the first to formally designate its national AI "
                  "supervisory authority, and on which date?"),
     "expected": None},   # unanswerable from this corpus — by design
]


def task_success(task, answer):
    """Mechanical success check: expected keyword in the final answer (None = unanswerable)."""
    if answer is None:
        return False
    if task["expected"] is None:
        # honest failure = admitting the corpus does not contain the answer
        return any(w in answer.lower() for w in ("cannot", "not available", "unable",
                                                 "no information", "unverifiable", "not find"))
    return task[___].lower() in answer.lower()     # which task field holds the ground truth?


print("Task suite ready:", [t["id"] for t in TASKS])

<details>
<summary><b>Click here for the solution</b></summary>

```python
    return task["expected"].lower() in answer.lower()
```

For the `loop_trap` task there is no ground truth in the corpus, so "success" means the agent
*admits* that — an honest `finish("… cannot be determined …")` counts, a confident guess does
not. Keep that definition in mind for Part C.

</details>

> **Q:** Restate Anthropic's workflow/agent distinction in terms of the "who decides the next step" question.
<details><summary>Click for answer</summary>

In <i>Building Effective Agents</i> (2024), workflows are systems where LLMs and tools are
orchestrated through <b>predefined code paths</b> — code decides the next step. Agents are
systems where the model <b>dynamically directs its own process</b> — the model decides the next
step. The distinction is exactly the question of where control resides, and it defines the
endpoints of the predictability–flexibility axis.
</details>

## Part C — A ReAct agent from scratch

We rebuild ReAct the way Yao et al. did in 2022, **before function-calling APIs existed**: the
format is taught by a **few-shot example trace** in the prompt, the model emits
`Thought n: … / Action n: …` as plain text, and the runtime parses the action line by string
matching, executes the tool, and pastes the **Observation** back into the context. The trace
*is* the agent's working memory — watch it grow.

Two runtime details that the paper's runtime also needed:

1. a **stop sequence** on `"Observation"` — otherwise the model happily *hallucinates* its own
   tool results instead of waiting for real ones;
2. a **step cap** — the loop is a cyclic graph, so termination is *not* guaranteed by
   construction (that is Part F's contrast).

In [ ]:
REACT_SYSTEM = """You are a careful research agent. You verify claims using ONLY these tools:

  search("query")   -> top matching pages from a fixed corpus (url, title, snippet)
  fetch("url")      -> the full text of one page
  finish("answer")  -> end the episode and return your final answer

Respond with exactly ONE step per turn, in this format and nothing else:

Thought <n>: <what do I know, what is missing, what next?>
Action <n>: <exactly one tool call>

Never write an Observation yourself - the runtime provides it after your action.
Prefer primary sources over news or blogs. If the corpus cannot answer the question,
call finish with an honest statement that the answer cannot be determined."""

FEW_SHOT = """Example episode (a different question, to show the format):

Question: In which year was the fictional Freedonia Data Act adopted?
Thought 1: I need an authoritative source, not an opinion piece. Search for it.
Action 1: search("Freedonia Data Act adoption year")
Observation 1: [{"url": "gov.example/data-act", "title": "Freedonia Data Act - official page", "snippet": "..."}]
Thought 2: An official government page is a primary source. Fetch it and find the year.
Action 2: fetch("gov.example/data-act")
Observation 2: "The Freedonia Data Act was adopted in 2019 and entered into force in 2020."
Thought 3: The adoption year 2019 is confirmed by a primary source. Done.
Action 3: finish("The Freedonia Data Act was adopted in 2019.")"""

ACTION_RE = re.compile(r'Action\s*\d*\s*:\s*(search|fetch|finish)\(\s*"(.*)"\s*\)')


def parse_action(text):
    """Extract (tool, argument) from the model's text — Yao et al.'s 'string matching'."""
    m = ACTION_RE.search(text)
    if m is None:
        return None, None
    return m.group(1), m.group(___)          # which capture group holds the argument?


# self-test (no LLM needed)
assert parse_action('Thought 1: hm.\nAction 1: search("EU AI Act dates")') == ("search", "EU AI Act dates")
assert parse_action('Action 3: finish("done")') == ("finish", "done")
assert parse_action("no action here") == (None, None)
print("Parser self-test passed.")

<details>
<summary><b>Click here for the solution</b></summary>

```python
    return m.group(1), m.group(2)
```

Group 1 is the tool name (`search|fetch|finish`), group 2 the quoted argument. Note how fragile
this is compared to the JSON-schema tool calls of Session 03 — one stray quote and the parse
fails. That fragility is historically authentic: it is exactly what function-calling APIs later
productised away. The control structure, however, is unchanged.

</details>

In [ ]:
MAX_OBS_CHARS = 600     # observations are truncated — context growth is ReAct's tax


def react_run(question, max_steps=8, verbose=True, detector=None):
    """The instrumented ReAct loop: returns answer, full trace, and cost statistics."""
    trace = f"Question: {question}\n"
    stats = {"llm_calls": 0, "prompt_tokens": 0, "output_tokens": 0, "latency_s": 0.0}
    actions = []                             # executed (tool, normalised-arg) pairs
    answer = None

    for step in range(1, ___ + 1):           # the hard step cap — every dynamic loop needs one
        messages = [
            {"role": "system", "content": REACT_SYSTEM},
            {"role": "user", "content": FEW_SHOT + "\n\nNow the real episode:\n\n"
                                        + trace + f"Thought {step}:"},
        ]
        raw, s = llm(messages, stop=[___])   # never let the model invent its own tool results
        stats["llm_calls"] += 1
        for key in ("prompt_tokens", "output_tokens", "latency_s"):
            stats[key] = round(stats[key] + s[key], 2)

        block = f"Thought {step}: {raw}"
        tool, arg = parse_action(___)        # parse the model's latest text

        if tool == "finish":
            answer = arg
            trace += block + "\n"
            if verbose:
                print(block)
            break

        norm = (tool, re.sub(r"\s+", " ", (arg or "").strip().lower()))
        if tool is None:
            observation = ("Invalid or missing action. Emit exactly one Action line using "
                           'search("..."), fetch("...") or finish("...").')
        elif detector is not None and detector(actions, tool, arg):
            observation = ("Loop detected: you already tried this exact action. Do NOT repeat "
                           "it. Change strategy, or finish(...) honestly with what you have.")
        elif tool == "search":
            observation = json.dumps(search(arg))
        elif tool == "fetch":
            observation = fetch(___)[:MAX_OBS_CHARS]     # the argument the model chose
        actions.append(norm)

        trace += block + f"\nObservation {step}: {observation}\n"
        if verbose:
            print(block)
            print(f"Observation {step}: {observation[:200]}…\n")

    stats["steps"] = len(actions) + (1 if answer is not None else 0)
    return {"answer": answer, "trace": trace, **stats}


print("react_run defined.")

<details>
<summary><b>Click here for the solution</b></summary>

```python
    for step in range(1, max_steps + 1):
        ...
        raw, s = llm(messages, stop=["Observation"])
        ...
        tool, arg = parse_action(raw)
        ...
            observation = fetch(arg)[:MAX_OBS_CHARS]
```

</details>

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- The prompt is rebuilt every iteration from the **entire trace so far** — the model has no
  memory between calls; the trace is the memory. That is why `prompt_tokens` grows every step:
  a ReAct run's token volume grows roughly quadratically with its length.
- `stop=["Observation"]` cuts generation the moment the model tries to write an observation
  itself — hallucinated tool results are the classic bug in hand-rolled ReAct runtimes.
- An unparseable action becomes an *observation about the error*, not an exception: the loop
  gives the model a chance to recover, which is the whole point of interleaving.
- `detector` is a hook for Part C's loop detector (report task R2): it sees the executed action
  history and the proposed action, and can veto execution by triggering a corrective
  observation instead.
- `stats` makes the run **instrumented**: calls, tokens, latency, steps — the raw material for
  Part E's comparison and for the tracing discipline of S11.

</details>

> **Q:** Describe the ReAct cycle and the function of each of its three elements.
<details><summary>Click for answer</summary>

The model alternates <b>thought, action, observation</b>. A thought is free-text reasoning —
never executed — in which the model assesses progress and commits to a next move. An action is a
tool invocation (<code>search</code>, <code>fetch</code>, <code>finish</code>) emitted as text
and executed by the runtime. An observation is the tool's result, appended to the context so the
next thought can interpret it. The cycle repeats until a <code>finish</code> action ends the
episode.
</details>

### C.1 — The clean run

Task `gpai_date` is well covered by the corpus. Read the trace top to bottom, like the lecture's
example: watch for a *source-selection policy* appearing in a thought (primary source vs news vs
blog) — a run-time quality decision no workflow author enumerated. If the misleading blog page
shows up in the search results, watch whether the agent takes the bait (**derailment**) or
reasons past it.

In [ ]:
if OLLAMA_OK:
    run_clean = react_run(TASKS[0]["question"], max_steps=___)   # a sensible cap for a bounded task
    print("\nFinal answer:", run_clean["answer"])
    print("Success:", task_success(TASKS[0], run_clean["answer"]))
    print({k: run_clean[k] for k in ("llm_calls", "steps", "prompt_tokens",
                                     "output_tokens", "latency_s")})
else:
    print("Skipping — Ollama not available.")

<details>
<summary><b>Click here for the solution</b></summary>

```python
    run_clean = react_run(TASKS[0]["question"], max_steps=8)
```

Any small cap (6–10) is fine — verification is a bounded task. A typical successful trace needs
3 steps: search → fetch the `europa.example` timeline → finish with "2 August 2025". If your run
fetched the blog page instead and reported a 2027 delay, you have just witnessed **derailment**:
one misleading observation bending the whole trajectory. Re-run the cell (sampling varies) and
compare traces — dynamic control means every run is a new path.

</details>

### C.2 — The loop trap

Task `loop_trap` asks for a fact the corpus deliberately does not contain. The lecture predicts
what a myopic loop does with it: **re-issuing a failed search with minor rewording, burning
budget**. Count the near-duplicate searches in the trace.

In [ ]:
if OLLAMA_OK:
    run_trap = react_run(TASKS[___]["question"], max_steps=6)    # the unanswerable task
    print("\nFinal answer:", run_trap["answer"])
    print("Honest failure:", task_success(TASKS[2], run_trap["answer"]))
    print({k: run_trap[k] for k in ("llm_calls", "steps", "prompt_tokens",
                                    "output_tokens", "latency_s")})
else:
    print("Skipping — Ollama not available.")

<details>
<summary><b>Click here for the solution</b></summary>

```python
    run_trap = react_run(TASKS[2]["question"], max_steps=6)
```

Typical behaviour: the agent searches, gets the vague `national-ai-authorities` blog page,
fetches it, learns nothing, searches again with a reworded query, gets the same page … until the
step cap fires or it gives up honestly. Compare `prompt_tokens` with the clean run: the trap run
pays more for *less* — unbounded cost is a billing event, which is why every production loop
ships with caps (steps, tokens, money, wall-clock).

</details>

> **📝 Report task R1:** Specify a full set of **stop conditions** for the ReAct claim-verification loop, grounded in what you observed in Part C (the clean run *and* the loop-trap run). Cover at least: a success condition, a hard step cap with a concrete number, a no-progress/stagnation condition, and a tool-failure policy — and say what the agent should return when it stops without an answer.
> *No solution is provided — include your stop-condition list and a short justification in your lab report.*

### C.3 — Loop detection (report task R2)

Caps are the backstop; **loop detection is the early exit**. The `react_run` loop already calls
an optional `detector(actions, tool, arg)` hook before executing an action: if it returns
`True`, the tool is *not* executed and the model receives a corrective observation instead.
Implement the detector.

In [ ]:
LOOP_WINDOW = 4      # how many recent actions to compare against


def repeated_action(actions, tool, arg):
    """Return True iff (tool, arg) already occurs among the last LOOP_WINDOW executed actions.
    Normalise the argument (lower-case, collapsed whitespace) before comparing — the executed
    history in `actions` is stored in exactly that normalised form."""
    # R2: implement the loop check.
    ___


if OLLAMA_OK:
    run_guarded = react_run(TASKS[2]["question"], max_steps=6, detector=repeated_action)
    print("\nFinal answer:", run_guarded["answer"])
    print("Honest failure:", task_success(TASKS[2], run_guarded["answer"]))
else:
    print("Skipping — Ollama not available.")

> **📝 Report task R2 (code):** Complete the cell above — the **loop detector**. It must compare each new action against the recent action history and, on a repeat, inject a corrective observation instead of executing the tool again (the lecture's mitigation list: step caps, **loop detection**, compaction). Re-run the loop-trap task and show in your report how the trace changes.
> *No solution is provided — include your code and the before/after traces in your lab report.*

> **Q:** Explain why *act-only* fails, and contrast its failure mode with *reason-only*'s.
<details><summary>Click for answer</summary>

Act-only issues tool calls without deliberating between them: it struggles to decompose the
goal, loses track of what has been established, and cannot reinterpret a disappointing result to
change strategy — it just acts again. Reason-only (CoT) fails by being <b>ungrounded</b>:
fluent, internally coherent hallucination on parametric memory alone, with no checkpoint where
reality can intervene. Act-only fails by being <b>undirected</b>; reason-only by being
ungrounded. The failures are complementary — which is the argument for interleaving: thoughts
ground actions in a plan, observations ground thoughts in the world.
</details>

> **Q (not exam-relevant):** On which task domains/benchmarks was ReAct originally evaluated?
<details><summary>Click for answer</summary>

Knowledge tasks: HotpotQA (multi-hop question answering) and FEVER (fact verification), using a
Wikipedia API. Interactive decision-making: ALFWorld (a text-based household environment) and
WebShop (simulated online shopping).
</details>

## Part D — Plan-and-Execute on the same tasks

Same goal, opposite decision rhythm: **decisions are front-loaded**. A **planner** call emits a
complete step list upfront — the **plan as artifact**, a JSON array you can print, log, inspect
and approve *before anything executes*. Then an **executor** runs one step at a time with
**narrow context** — its step and its inputs, not the history:

- `search` and `fetch` steps execute as **plain code — no LLM call at all**,
- only the final `finish` step calls the model, and it sees just the fetched pages.

Because the planner cannot know URLs in advance, plans reference earlier results
(`"$2"` = top URL found by the search in step 2). When a step fails — a 404 from a guessed URL,
an empty result list — control returns to the planner with the failure as feedback:
**replanning is an explicit, observable event**, not silent drift.

(On the strong/cheap split: in production the planner would be your strongest reasoning model
and the executor a cheap one — the S05 barbell. With one local model we emulate the split by
giving the planner a low temperature and the executor no model at all where code suffices.)

In [ ]:
PLANNER_SYSTEM = """You are the planner of a research agent. Write a COMPLETE plan to answer
the user's question, as a JSON array of steps. Allowed step types:

  {"tool": "search", "input": "<query>"}                   -> keyword search over a fixed corpus
  {"tool": "fetch",  "input": "$k"}                        -> fetch the top URL found by step k
  {"tool": "finish", "input": "<what to extract/answer>"}  -> read fetched pages, give the answer

Rules: you know NO URLs in advance, so every fetch must reference a search step (e.g. "$1").
Prefer primary sources. Use 3 to 5 steps, ending with exactly one finish step.
Respond with the JSON array and NOTHING else."""


def make_plan(question, feedback=""):
    """One planner call. Returns (plan, stats). `feedback` carries execution failures back in."""
    user = f"Question: {question}"
    if feedback:
        user += ("\n\nA previous plan failed during execution. Feedback:\n" + feedback
                 + "\nWrite a revised, complete plan.")
    raw, s = llm([{"role": "system", "content": PLANNER_SYSTEM},
                  {"role": "user", "content": ___}],       # the planner sees goal + feedback only
                 temperature=0.1, num_predict=400)
    m = re.search(r"\[.*\]", raw, re.S)                    # tolerate prose around the JSON array
    try:
        plan = json.loads(m.group(___)) if m else []       # the whole regex match
    except json.JSONDecodeError:
        plan = []
    return plan, s


if OLLAMA_OK:
    demo_plan, demo_stats = make_plan(TASKS[0]["question"])
    print(json.dumps(demo_plan, indent=2))
    print(demo_stats)
else:
    print("Skipping — Ollama not available.")

<details>
<summary><b>Click here for the solution</b></summary>

```python
    raw, s = llm([{"role": "system", "content": PLANNER_SYSTEM},
                  {"role": "user", "content": user}],
                 temperature=0.1, num_predict=400)
    ...
        plan = json.loads(m.group(0)) if m else []
```

</details>

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- The plan is now a **first-class object in your program**: print it, diff it, count its steps,
  refuse to execute it, show it to a human for approval. None of that exists in ReAct, where
  "the plan" lives implicitly inside thoughts and is only reconstructable forensically.
- `m.group(0)` is the whole regex match — we grab the outermost `[...]` so a chatty model that
  wraps the JSON in prose still parses. A malformed plan becomes an empty list, which the outer
  loop treats as a failure and feeds back to the planner.
- `temperature=0.1`: planning errors compound through every downstream step, so the planner is
  the one place where we want determinism, not creativity.

</details>

In [ ]:
def execute_step(i, step, memory, stats, verbose=True):
    """Execute ONE plan step. `search`/`fetch` are pure code (zero LLM calls); only `finish`
    calls the model — with narrow context. Returns (result, failure); a non-None failure
    string aborts execution and becomes the replanner's feedback."""
    tool, arg = step.get("tool"), str(step.get("input", ""))

    if tool == "search":
        hits = search(arg)
        memory["hits"][i] = hits
        if verbose:
            print(f"  step {i}: search({arg!r}) -> {[h['url'] for h in hits]}")
        return (hits, None) if hits else (None, f"step {i}: search({arg!r}) found nothing")

    if tool == "fetch":
        ref = re.match(r"\$(\d+)", arg)
        if ref and memory["hits"].get(int(ref.group(1))):
            url = memory["hits"][int(ref.group(1))][0]["url"]
        else:
            url = arg                          # the planner guessed a URL — plans can go stale
        text = fetch(url)
        if verbose:
            print(f"  step {i}: fetch({url!r})")
        if text.startswith(___):               # a dead link is a replan trigger, not a crash
            return None, f"step {i}: {text}"
        memory["pages"].append((url, text))
        return url, None

    if tool == "finish":
        context = "\n\n".join(f"SOURCE {u}:\n{t}" for u, t in memory["pages"])
        prompt = (f"Task: {arg}\n\nUse ONLY these sources:\n\n{context}\n\n"
                  "If the sources do not contain the answer, state clearly that it cannot be "
                  "determined from the available sources. Answer in one or two sentences.")
        answer, s = llm([{"role": "user", "content": prompt}], num_predict=200)
        stats["llm_calls"] += 1
        for key in ("prompt_tokens", "output_tokens", "latency_s"):
            stats[key] = round(stats[key] + s[key], 2)
        return answer, None

    return None, f"step {i}: unknown tool {tool!r}"


def plan_and_execute(question, max_replans=2, verbose=True):
    """The outer loop: plan -> execute step by step -> on failure, replan with feedback."""
    stats = {"llm_calls": 0, "prompt_tokens": 0, "output_tokens": 0, "latency_s": 0.0,
             "steps": 0}
    feedback, answer, replans = "", None, 0
    for attempt in range(___ + 1):             # planning attempts are bounded too
        plan, s = make_plan(question, feedback)
        stats["llm_calls"] += 1
        for key in ("prompt_tokens", "output_tokens", "latency_s"):
            stats[key] = round(stats[key] + s[key], 2)
        if verbose:
            print(f"PLAN (attempt {attempt + 1}): {json.dumps(plan)}")
        memory = {"hits": {}, "pages": []}
        failure = None if plan else "the planner returned no parseable JSON plan"
        for i, step in enumerate(plan, start=1):
            result, failure = execute_step(i, step, memory, stats, verbose)
            stats["steps"] += 1
            if failure:
                break
            if step.get("tool") == ___:        # which step type produces the final answer?
                answer = result
        if failure is None:
            break                              # the plan executed to completion
        feedback, answer, replans = failure, None, replans + 1
        if verbose:
            print(f"  REPLAN EVENT: {failure}")
    return {"answer": answer, "replans": replans, **stats}


print("plan_and_execute defined.")

<details>
<summary><b>Click here for the solution</b></summary>

```python
        if text.startswith("ERROR"):
    ...
    for attempt in range(max_replans + 1):
    ...
            if step.get("tool") == "finish":
```

</details>

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- **Zero-LLM steps:** `search` and `fetch` never touch the model. In ReAct, *every* control
  decision costs one LLM call carrying the whole trace; here the mechanics run as plain code —
  that is where the token savings in Part E will come from.
- **Narrow context:** the one LLM call (`finish`) sees only the fetched pages and its
  instruction — not a growing history. Cost per call is near-constant, and the executor cannot
  be derailed by trace garbage it never sees.
- **Replanning is observable:** a failed step does not crash and is not silently patched — it
  ends the attempt, is printed as a `REPLAN EVENT`, and returns to the planner as feedback.
  Count these events and you have a drift metric; ReAct's changes of mind are invisible.
- `max_replans` bounds the outer loop just as `max_steps` bounds ReAct — every loop in an
  agentic system needs an explicit budget, whichever regime owns it.

</details>

In [ ]:
if OLLAMA_OK:
    print("=== clean task ===")
    pe_clean = plan_and_execute(TASKS[0]["question"])
    print("Final answer:", pe_clean["answer"])
    print("Success:", task_success(TASKS[0], pe_clean["answer"]), "|",
          {k: pe_clean[k] for k in ("llm_calls", "steps", "replans",
                                    "prompt_tokens", "output_tokens", "latency_s")})

    print("\n=== loop-trap task ===")
    pe_trap = plan_and_execute(TASKS[___]["question"])       # the unanswerable task again
    print("Final answer:", pe_trap["answer"])
    print("Honest failure:", task_success(TASKS[2], pe_trap["answer"]), "|",
          {k: pe_trap[k] for k in ("llm_calls", "steps", "replans",
                                   "prompt_tokens", "output_tokens", "latency_s")})
else:
    print("Skipping — Ollama not available.")

<details>
<summary><b>Click here for the solution</b></summary>

```python
    pe_trap = plan_and_execute(TASKS[2]["question"])
```

Note what the trap does — and does not — cost here. The plan's length bounds the work: the
executor runs its 3–5 steps, the `finish` call reads the (unhelpful) pages and reports that the
answer cannot be determined. **The executor cannot loop, because it has no vote on the next
step.** Compare that with the ReAct trap run in C.2, which burned its whole step budget on
rewordings. The trade arrives in the other direction: if the planner guessed a URL instead of
using `"$k"`, you will see a 404 → `REPLAN EVENT` — the stale-plan failure mode.

</details>

> **Q:** Why does the executor's context stay small in Plan-and-Execute, and what two benefits follow?
<details><summary>Click for answer</summary>

Because the executor is given only its current step and the inputs that step needs — not the
accumulated trajectory. Benefit one: <b>cost</b> — short, near-constant prompts per step instead
of a linearly growing history, and a cheaper model (or plain code, as in our
<code>search</code>/<code>fetch</code> steps) often suffices for the narrow task. Benefit two:
<b>robustness</b> — the executor cannot be distracted or derailed by irrelevant global context
it never sees; the information architecture enforces focus.
</details>

## Part E — Two regimes, side by side

Now the measurement the lecture's comparison table promised. We run **both regimes on all three
tasks** with identical tools and collect the instrumentation. Read the numbers along the
table's axes:

- **LLM calls / steps** — who spends model calls on control?
- **prompt tokens** — ReAct re-sends its whole growing trace every step; the P&E executor sees a
  near-constant narrow context (and `search`/`fetch` cost zero LLM calls),
- **latency shape** — ReAct's time accumulates step by step; P&E waits up front in the planner,
- **failure modes** — looping vs stale plans: watch the `loop_trap` row and the `replans` count.

Local-model measurements are noisy — re-run the cell if a run derails; the *shape* of the
comparison is what matters, and interpreting it is report task R3.

In [ ]:
if OLLAMA_OK:
    RUNNERS = {
        "ReAct": lambda q: react_run(q, max_steps=8, verbose=False),
        "Plan-and-Execute": lambda q: plan_and_execute(q, verbose=False),
    }
    rows = []
    for task in TASKS:
        for regime, runner in RUNNERS.items():
            r = runner(task["question"])
            rows.append({"task": task["id"], "regime": regime,
                         "llm_calls": r["llm_calls"], "steps": r["steps"],
                         "prompt_tokens": r["prompt_tokens"],
                         "output_tokens": r["output_tokens"],
                         "latency_s": r["latency_s"],
                         "success": task_success(task, r[___])})   # the run's final answer
    df = pd.DataFrame(___)                     # rows -> table
    print(df.to_string(index=False))
else:
    print("Skipping — Ollama not available.")

<details>
<summary><b>Click here for the solution</b></summary>

```python
                         "success": task_success(task, r["answer"])})
    df = pd.DataFrame(rows)
```

</details>

In [ ]:
if OLLAMA_OK:
    agg = df.groupby(___)[["llm_calls", "steps", "prompt_tokens",
                           "output_tokens", "latency_s"]].sum()
    print(agg, "\n")
    print("success by task:")
    print(df.pivot(index="task", columns="regime", values="success"))

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    df.pivot(index="task", columns="regime", values=___).plot.bar(ax=axes[0], rot=0)
    axes[0].set_title("LLM calls per task")
    df.pivot(index="task", columns="regime", values="prompt_tokens").plot.bar(ax=axes[1], rot=0)
    axes[1].set_title("Prompt tokens per task")
    plt.tight_layout()
    plt.show()
else:
    print("Skipping — Ollama not available.")

<details>
<summary><b>Click here for the solution</b></summary>

```python
    agg = df.groupby("regime")[["llm_calls", "steps", "prompt_tokens",
                                "output_tokens", "latency_s"]].sum()
    ...
    df.pivot(index="task", columns="regime", values="llm_calls").plot.bar(ax=axes[0], rot=0)
```

Typical shape: ReAct's `prompt_tokens` bar dwarfs P&E's on every task (the growing trace is
re-sent every step), most extremely on `loop_trap`. P&E's `llm_calls` sit at 2–3 regardless of
task. If a P&E run shows `replans > 0`, its planner paid twice — the stale-plan tax.

</details>

> **📝 Report task R3:** Using your Part E table, compare ReAct and Plan-and-Execute on this lab's verification tasks along four axes: **LLM calls / steps**, **token cost**, **latency shape** (where the waiting happens), and **failure modes you observed**. Then decide which regime you would ship for the research agent's *claim-verification* subtask and justify the choice with rows of the lecture's three-regimes table (predictability, flexibility, debuggability, cost).
> *No solution is provided — include your table, answer and a short justification in your lab report.*

> **Q:** Compare the cost profiles of ReAct and Plan-and-Execute on a 20-step task.
<details><summary>Click for answer</summary>

ReAct: 20 model calls, each carrying the full growing history, so token volume grows roughly
quadratically with steps; cost is unknown until the run ends and unbounded without caps.
Plan-and-Execute: one or two heavy planner calls plus 20 light executor calls with small,
near-constant context; total cost is estimable once the plan exists. The shape is the S05
barbell — expensive deliberation at the edges, cheap mechanics between. Frequent replanning
erodes the advantage: replanning after every step pays planner prices for ReAct behaviour.
</details>

## Part F — The static end: a DAG for a fixed subtask

The research agent's *"EU AI Act key facts"* sub-report always has the same structure: two
independent branches (timeline, penalties), each search → fetch → extract, then one draft node
that merges. Nothing about that structure depends on run-time information — so **code owns the
control flow** and the model only fills node content; it has no vote on which node runs next.
We write the graph explicitly: **nodes are steps, directed edges are data dependencies,
execution follows topological order.**

Two properties fall out of the structure itself, before any node runs:

- **Termination by construction** — acyclic means no path returns to an earlier node, so every
  node executes at most once (contrast: `react_run` needed `max_steps` as a bolted-on cap),
- **Parallelism for free** — nodes with no directed path between them are provably independent;
  our two branches could run concurrently without any synchronisation logic.

In [ ]:
def topological_waves(dag):
    """Group nodes into 'waves': every node in a wave has all its dependencies satisfied by
    earlier waves — so the nodes of one wave are mutually independent and could run in
    parallel, for free. The graph itself is the proof of non-interference."""
    done, waves = set(), []
    while len(done) < len(dag):
        wave = sorted(n for n, spec in dag.items()
                      if n not in done and all(d in ___ for d in spec["deps"]))
        if not wave:
            raise ValueError("cycle detected — this graph is not a DAG")
        waves.append(wave)
        done.update(___)                       # these nodes now count as satisfied dependencies
    return waves


def run_dag(dag, verbose=True):
    """Execute a DAG in topological order. Each node's function receives a dict with exactly
    its dependencies' results — nodes never see anything else (narrow context by construction)."""
    results = {}
    for wave in topological_waves(dag):
        if verbose:
            print("wave (parallel for free):", wave)
        for name in wave:
            deps = {d: results[d] for d in dag[name]["deps"]}
            results[name] = dag[name]["fn"](___)
    return results


# structural self-test (no LLM): a diamond graph has 3 waves, with a parallel middle
_diamond = {"a": {"deps": [], "fn": lambda d: 1},
            "b": {"deps": ["a"], "fn": lambda d: d["a"] + 1},
            "c": {"deps": ["a"], "fn": lambda d: d["a"] + 2},
            "d": {"deps": ["b", "c"], "fn": lambda d: d["b"] + d["c"]}}
assert topological_waves(_diamond) == [["a"], ["b", "c"], ["d"]]
assert run_dag(_diamond, verbose=False)["d"] == 5
print("DAG engine self-test passed.")

<details>
<summary><b>Click here for the solution</b></summary>

```python
        wave = sorted(n for n, spec in dag.items()
                      if n not in done and all(d in done for d in spec["deps"]))
        ...
        done.update(wave)
        ...
            results[name] = dag[name]["fn"](deps)
```

</details>

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- This is Kahn's algorithm in wave form: repeatedly take every node whose dependencies are all
  `done`. If a pass finds no such node while nodes remain, some node depends (transitively) on
  itself — a cycle — and we refuse to run: **termination is checked structurally, before
  execution**, not enforced at run time by a counter.
- Each wave's nodes are mutually independent *by construction of the edge set*; a real engine
  would hand a wave to a thread pool. We run them sequentially for readability — the guarantee
  is the point, not the speed-up.
- `run_dag` passes each node **only** its dependencies' results. Notice the family resemblance
  to the P&E executor's narrow context — at the static end, the graph enforces it perfectly.

</details>

In [ ]:
def llm_node(template):
    """Wrap an LLM call as a DAG node: the prompt sees ONLY the node's dependency results."""
    def fn(deps):
        context = "\n\n".join(str(v)[:800] for v in deps.values())
        text, _ = llm([{"role": "user", "content": template.format(context=context)}],
                      num_predict=250)
        return text
    return fn


RESEARCH_DAG = {
    "search_timeline":   {"deps": [],
                          "fn": lambda d: search("EU AI Act implementation timeline application dates")},
    "search_penalties":  {"deps": [],
                          "fn": lambda d: search("EU AI Act penalties maximum fine")},
    "fetch_timeline":    {"deps": ["search_timeline"],
                          "fn": lambda d: fetch(d[___][0]["url"])},      # top hit of its search
    "fetch_penalties":   {"deps": ["search_penalties"],
                          "fn": lambda d: fetch(d["search_penalties"][0]["url"])},
    "extract_timeline":  {"deps": ["fetch_timeline"],
                          "fn": llm_node("Extract the application dates of the EU AI Act as a "
                                         "short bullet list from this page:\n\n{context}")},
    "extract_penalties": {"deps": ["fetch_penalties"],
                          "fn": llm_node("Extract the maximum fines of the EU AI Act as a "
                                         "short bullet list from this page:\n\n{context}")},
    "draft":             {"deps": [___, "extract_penalties"],           # the draft needs both branches
                          "fn": llm_node("Write a five-sentence 'EU AI Act key facts' paragraph "
                                         "from these notes:\n\n{context}")},
}

print("Execution waves:", topological_waves(RESEARCH_DAG))

if OLLAMA_OK:
    out = run_dag(RESEARCH_DAG)
    print("\n--- draft ---")
    print(out["draft"])
else:
    print("Ollama not available — structure shown above, node execution skipped.")

<details>
<summary><b>Click here for the solution</b></summary>

```python
    "fetch_timeline":    {"deps": ["search_timeline"],
                          "fn": lambda d: fetch(d["search_timeline"][0]["url"])},
    ...
    "draft":             {"deps": ["extract_timeline", "extract_penalties"], ...}
```

Expected waves: `[['search_penalties', 'search_timeline'], ['fetch_penalties',
'fetch_timeline'], ['extract_penalties', 'extract_timeline'], ['draft']]` — the two branches
travel side by side (parallel for free), and `draft` is the fan-in. Run it twice: the *path* is
identical every time; only node contents vary. That is what "static" means — and exactly what
you could never say about the ReAct traces of Part C. This graph is Anthropic's **prompt
chaining + parallelization (sectioning)** in seven nodes; a production version would add a
**gate** after each extract node (non-empty? dates present?) before `draft` consumes it.

</details>

> **📝 Report task R4:** Design the **hybrid**: take your Part F DAG as the outer skeleton of the research agent and state where exactly you would embed a bounded dynamic element (ReAct loop), and why *there* and nowhere else. Give the cap you would set and one gate you would place after the dynamic node.
> *No solution is provided — include your design and a short justification in your lab report.*

> **Q:** Define a DAG in the orchestration context and explain why acyclicity guarantees termination.
<details><summary>Click for answer</summary>

A directed acyclic graph whose nodes are steps (LLM calls, tools, functions) and whose directed
edges are data dependencies; execution proceeds in topological order. Acyclicity means no path
returns to an earlier node, so each node executes at most once; with finitely many nodes, every
run necessarily ends. Termination is a <b>structural</b> property — no step counter or watchdog
needed, unlike in cyclic (loop-based) regimes: <code>react_run</code> needed
<code>max_steps</code>; <code>run_dag</code> needs nothing.
</details>

## Part G — Tuning & exploration

No gaps in this part — knobs to turn and re-run. Every knob moves the system along the axes of
the three-regimes table (predictability, flexibility, cost); note what changes in the traces
and in the stats.

In [ ]:
def tuning_run(task_idx=2, max_steps=6, obs_chars=600, use_detector=False):
    """Re-run one task under different knobs (no gaps here — just play).

    task_idx      0 = gpai_date, 1 = max_fine, 2 = loop_trap
    max_steps     the ReAct hard cap — try 3 vs 12 on the loop trap
    obs_chars     observation truncation — 200 loses evidence, 2000 pays more tokens
    use_detector  plug in your R2 loop detector (only after you implemented it!)
    """
    global MAX_OBS_CHARS
    old = MAX_OBS_CHARS
    MAX_OBS_CHARS = obs_chars
    try:
        det = repeated_action if use_detector else None
        r = react_run(TASKS[task_idx]["question"], max_steps=max_steps,
                      verbose=False, detector=det)
    finally:
        MAX_OBS_CHARS = old
    print(f"task={TASKS[task_idx]['id']}  answer={str(r['answer'])[:90]!r}")
    print({k: r[k] for k in ("llm_calls", "steps", "prompt_tokens",
                             "output_tokens", "latency_s")})


if OLLAMA_OK:
    try:
        import ipywidgets as widgets
        widgets.interact_manual(tuning_run,
                                task_idx=(0, 2), max_steps=(2, 12),
                                obs_chars=(100, 2000, 100), use_detector=False)
    except ImportError:
        print("ipywidgets not installed — calling tuning_run() directly instead:\n")
        tuning_run(task_idx=2, max_steps=6)
else:
    print("Skipping — Ollama not available.")

# Other knobs worth editing directly in earlier cells:
#  - temperature in llm(): 0.0 makes ReAct traces (almost) repeatable; 0.8 derails more often
#  - k in search(): with k=1 the agent never sees alternative sources — derailment risk rises
#  - LOOP_WINDOW: 1 misses slow loops; 6 may flag legitimate revisits
#  - max_replans in plan_and_execute(): 0 turns P&E into a strict one-shot plan

## Wrap-up

**Takeaways**

- **Orchestration is the control layer** — the logic deciding which step executes next. The one
  question: *who decides the next step, your code or the model?* Predictability and flexibility
  cannot both be maximal — a logical trade-off, not a technological one.
- **ReAct** interleaves Thought / Action / Observation: thoughts ground actions in a plan,
  observations ground thoughts in the world. Grounded and adaptive — but myopic, loop-prone,
  derailable, and the growing trace is both its working memory and its bill. Every dynamic loop
  ships with caps, loop detection, and full tracing (S11).
- **Plan-and-Execute** front-loads decisions into a plan artifact, executes with narrow context
  (often without any model call), and makes replanning an explicit, observable event. Weakness:
  plans go stale in volatile environments.
- **Static DAGs** put code fully in charge: termination by construction (acyclicity),
  parallelism for free, unit-testable nodes, fixed cost. Anthropic's five workflow patterns —
  chaining, routing, parallelization, orchestrator-workers, evaluator-optimizer — are the
  vocabulary of this range.
- **No regime dominates — the task's predictability decides, not fashion.** Use the simplest
  pattern that works; add autonomy only when measured.

**Next week (S07):** Frameworks vs plain code — LangChain, LangGraph & co., and when a
framework earns its abstraction versus the plain-Python loops you built today.

**For your lab report**

- [ ] **R1** — stop-condition set for the ReAct verifier, grounded in your Part C traces
- [ ] **R2** — loop-detector code + before/after traces on the loop-trap task (Part C.3)
- [ ] **R3** — regime comparison along the table's axes + your shipping decision (Part E)
- [ ] **R4** — hybrid design: DAG skeleton with one bounded dynamic node + gate (Part F)